#Prerequisites

### Load 'employee.csv' into DataFrame

In [0]:
employee_df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sendbox/dataset-20260601T053440Z-3-001/dataset/employee.csv",
    header=True,
    inferSchema=True,
    sep="|",
    quote= "'",
    
)
employee_df.display()

# Handling Missing Records

### Dropping Missing Records

In [0]:
employee_df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sendbox/dataset-20260601T053440Z-3-001/dataset/employee.csv",
    header=True,
    inferSchema=True,
    sep="|",
    quote= "'",
)
#employee_df.na.drop(how="any").display()
#employee_df.na.drop(how="all").display()
employee_df.na.drop(subset=["id","name"]).display()

In [0]:
from pyspark.sql.functions import col
employee_df.filter(col("id").isNull()).select("name").display()

In [0]:
from pyspark.sql.functions import col
employee_df.na.drop(how="all",subset=["id","name"]).display()

### Filling Missing Records

In [0]:
employee_df.na.fill("NULL IN SOURCE").display()

### Approach 01

In [0]:
employee_df.na.fill("NULL IN SOURCE", subset=["name", "gen"]).na.fill(
    -1, subset=["id"]
).na.fill(0, subset=["exp"]).display()

### Approach 02

In [0]:
MISSING_DEFAULT_VALUES = {
    "name":"Anonymous",
    "id":-1,
    "exp":0
}
employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

In [0]:
from pyspark.sql.functions import when

employee_df.withColumn(
    "name", when(col("name").isNull(), "Anonymous").otherwise(col("name"))
).display()

In [0]:
from pyspark.sql.functions import current_date
MISSING_DEFAULT_VALUES = {
    "name":"Anonymous",
    "id":-1,
    "exp":0,
    "doj":current_date()
}
employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

In [0]:
from pyspark.sql.functions import when, current_date, col

employee_df.withColumn(
    "doj", when(col("doj").isNull(), current_date()).otherwise(col("doj"))
).display()

In [0]:
from pyspark.sql.functions import avg
avg_exp = employee_df.select(avg("exp")).collect()[0][0]
print(avg_exp)


In [0]:
from pyspark.sql.functions import current_date
MISSING_DEFAULT_VALUES = {
    "name":"Anonymous",
    "id":-1,
    "exp":round(avg_exp)
}
employee_df.na.fill(MISSING_DEFAULT_VALUES).display()

In [0]:
employee_df.withColumn("gen", when(col("gen") == "M", "Male")).withColumn(
    "gen", when(col("gen") == "F", "Female")
).withColumn("gen", when(col("gen") == "T", "Transgender")).display()

In [0]:
# employee_df.na.replace(-1,1,subset=["exp"]).display()
employee_df.na.replace({
    "M":"Male",
    "F":"Female",
    "T":"Transgender"
},subset=["gen"]).display()

In [0]:
from pyspark.sql.functions import avg
avg_exp = employee_df.select(avg("exp")).collect()[0][0]
mode_gender = employee_df.groupBy("gen").count().sort(col("count"),ascending=False).display()

In [0]:
#employee_df.na.replace(-1,1,subset=["exp"]).display()
employee_df.na.replace({
    "M":"Male",
    "F":"Female",
    "T":"Transgender"
}).display()

